In [ ]:
!pip install pillow==10.2.0

In [ ]:
# !pip install -U ultralytics huggingface_hub torch torchvision torchaudio opencv-python matplotlib tqdm pyyaml numpy


In [ ]:
!pip install ultralytics huggingface_hub torch torchvision torchaudio opencv-python matplotlib tqdm pyyaml numpy

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")


In [ ]:
# download dataset từ Google Drive
!pip install -q gdown

import gdown
import zipfile
import os

FILE_ID = "1bvFqlR1xyChacZ8O2uVn7l8TtD3-ezPX"
ZIP_PATH = "/content/dataset.zip"
EXTRACT_PATH = "/content/dataset"

# download
gdown.download(f"https://drive.google.com/uc?id={FILE_ID}", ZIP_PATH, quiet=False)

# unzip
os.makedirs(EXTRACT_PATH, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("✅ Dataset ready:", EXTRACT_PATH)

Downloading...
From (original): https://drive.google.com/uc?id=1bvFqlR1xyChacZ8O2uVn7l8TtD3-ezPX
From (redirected): https://drive.google.com/uc?id=1bvFqlR1xyChacZ8O2uVn7l8TtD3-ezPX&confirm=t&uuid=fa9a52f3-e83a-4266-802f-553b1f7420c9
To: /content/dataset.zip
100%|██████████| 2.41G/2.41G [01:00<00:00, 40.0MB/s]


✅ Dataset ready: /content/dataset


In [ ]:
import PIL
print("Pillow:", PIL.__version__)

import torch
print("Torch:", torch.__version__)

from ultralytics import YOLO
print("YOLO import OK")


Pillow: 10.2.0
Torch: 2.10.0+cu128
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
YOLO import OK


In [ ]:
# DATA_ROOT = "/content/drive/MyDrive/DAT/CAMO_Dataset_Full"
DATA_ROOT = "/content/dataset/COD10K-SMM"
PROJECT_DIR = "DAT"
RUN_NAME = "yolov8_camo_fs"


In [ ]:
import os

assert os.path.exists(f"{DATA_ROOT}/data.yaml"), "❌ Không thấy data.yaml"
assert os.path.exists(f"{DATA_ROOT}/train/images"), "❌ Thiếu train/images"
assert os.path.exists(f"{DATA_ROOT}/train/labels"), "❌ Thiếu train/labels"
assert os.path.exists(f"{DATA_ROOT}/val/images"), "❌ Thiếu val/images"
assert os.path.exists(f"{DATA_ROOT}/val/labels"), "❌ Thiếu val/labels"

print("✅ Dataset trong Drive OK")


✅ Dataset trong Drive OK


In [ ]:
import yaml

# Đường dẫn tới file yaml trong môi trường Colab
yaml_path = f"{DATA_ROOT}/data.yaml"

# Đọc file yaml
with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

# Sửa lại đường dẫn path cho đúng với vị trí giải nén thực tế
data['path'] = DATA_ROOT # Sẽ là "/content/dataset"
data['train'] = "train/images"
data['val'] = "val/images"

# Ghi lại file yaml
with open(yaml_path, 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

print(f"✅ Đã cập nhật lại đường dẫn trong {yaml_path}")


✅ Đã cập nhật lại đường dẫn trong /content/dataset/COD10K-SMM/data.yaml


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

results = model.train(
    data=f"{DATA_ROOT}/data.yaml",
    epochs=100,
    imgsz=640,
    batch=64,
    # patience=30,
    optimizer="AdamW",
    lr0=1e-4,
    device=0,
    workers=4,
    project=PROJECT_DIR,
    name=RUN_NAME,
    verbose=True,
    plots=False,
    seed=42,
    deterministic=True,
    # 🔥 phần QUAN TRỌNG
    auto_augment=None,
    close_mosaic=0,

    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.0,

    degrees=0.0,
    translate=0.0,
    scale=0.0,
    shear=0.0,
    perspective=0.0,

    fliplr=0.0,
    flipud=0.0,

    mosaic=0.0,
    mixup=0.0,
    copy_paste=0.0,
    erasing=0.0,
    augmentations=[],
)

Ultralytics 8.4.50 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, augmentations=[], auto_augment=None, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=0, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/COD10K-SMM/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.0, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=yolov8_camo_fs, nbs=64, nms=False, opset=None, optimize=False, optimizer=

In [ ]:
from ultralytics import YOLO

BEST_WEIGHTS = "/content/runs/detect/DAT/yolov8_camo_fs/weights/best.pt"

assert os.path.exists(BEST_WEIGHTS), f"❌ Không thấy {BEST_WEIGHTS}"

model = YOLO(BEST_WEIGHTS)

metrics = model.val(
    data=f"{DATA_ROOT}/data.yaml",
    split="val",
    verbose=False,
    plots=False
)

print(f"mAP@50    : {metrics.box.map50:.4f}")
print(f"mAP@50-95 : {metrics.box.map:.4f}")


Ultralytics 8.4.50 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
Model summary (fused): 73 layers, 11,152,287 parameters, 0 gradients, 28.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3004.5±1354.6 MB/s, size: 197.4 KB)
val: Scanning /content/dataset/COD10K-SMM/val/labels.cache... 2026 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2026/2026 849.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 127/127 16.0it/s 8.0s
                   all       2026       2365      0.586      0.149      0.166      0.107
Speed: 0.4ms preprocess, 1.5ms inference, 0.0ms loss, 0.6ms postprocess per image
mAP@50    : 0.1662
mAP@50-95 : 0.1068


In [ ]:
model.predict(
    source=f"{DATA_ROOT}/val/images",
    conf=0.25,
    save=True
)



WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/2026 /content/dataset/COD10K-SMM/val/images/COD10K-CAM-1-Aquatic-1-BatFish-2.jpg: 448x640 (no detections), 70.0ms
image 2/2026 /content/dataset/COD10K-SMM/val/images/COD10K-CAM-1-Aquatic-1-BatFish-4.jpg: 448x640 (no detections), 6.9ms
image 3/2026 /content/dataset/COD10K-SMM/val/images/COD10K-CAM-1-Aquatic-1-BatFish-5.jpg: 480x640 (no detections), 69.5ms
image 4/2026 /content/dataset/COD10K-SMM/val/images/COD10K-CAM-1-Aquatic-1-BatFish-6.jpg: 448

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'Grasshopper', 1: 'Heron', 2: 'Katydid', 3: 'Dog', 4: 'Spider', 5: 'SeaHorse', 6: 'Human', 7: 'Dragonfly', 8: 'Butterfly', 9: 'Owl', 10: 'Fish', 11: 'StarFish', 12: 'Lizard', 13: 'Mantis', 14: 'Pipefish', 15: 'Bee', 16: 'Moth', 17: 'StickInsect', 18: 'Deer', 19: 'BatFish', 20: 'Gecko', 21: 'Tiger', 22: 'Duck', 23: 'Pagurian', 24: 'Flounder', 25: 'Frogmouth', 26: 'Snake', 27: 'Crab', 28: 'Caterpillar', 29: 'Mockingbird', 30: 'Rabbit', 31: 'Frog', 32: 'Toad', 33: 'Octopus', 34: 'GhostPipefish', 35: 'Lion', 36: 'Bird', 37: 'Wolf', 38: 'Centipede', 39: 'Leopard', 40: 'Cicada', 41: 'Ant', 42: 'Chameleon', 43: 'Slug', 44: 'Worm', 45: 'Shrimp', 46: 'ScorpionFish', 47: 'Cat', 48: 'Crocodile', 49: 'Turtle', 50: 'Bug', 51: 'Kangaroo', 52: 'Grouse', 53: 'Sciuridae', 54: 'Owlfly', 55: 'Cheetah', 56: 'Other', 57: 'Bittern', 58: 'Giraffe', 59: 'FrogF

In [ ]:
import shutil, os

PROJECT_DIR = "/content/runs/detect/DAT"
RUN_NAME = "yolov8_camo_fs"

# dst = f"/content/drive/MyDrive/DAT/{RUN_NAME}"

# if os.path.exists(dst):
#     shutil.rmtree(dst)

# shutil.copytree(f"{PROJECT_DIR}/{RUN_NAME}", dst)

# print("✅ Đã copy model về Drive:", dst)



In [ ]:
from huggingface_hub import login, HfApi

login(token="YOUR_HUGGINGFACE_TOKEN")  # dán token WRITE

api = HfApi()
repo_id = "Thanhdat3010/yolov8s-COD10K-SMM"  # đổi nếu cần

api.create_repo(repo_id=repo_id, exist_ok=True, repo_type="model")

api.upload_folder(
    folder_path=f"{PROJECT_DIR}/{RUN_NAME}",
    repo_id=repo_id,
    repo_type="model"
)

print("✅ Push HuggingFace xong:", repo_id)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...8_camo_fs/weights/best.pt:  65%|######4   | 14.6MB / 22.6MB            

  ...8_camo_fs/weights/last.pt:   2%|1         |  390kB / 22.6MB            

✅ Push HuggingFace xong: Thanhdat3010/yolov8s-COD10K-SMM
